#### Import

In [14]:
from pdf2image import convert_from_path
import openai 
from openai import OpenAI
import json
import re
from dotenv import load_dotenv
import os
from typing import List, Tuple , Union
import numpy as np
import logging
import time
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import as_completed
import easyocr
reader = easyocr.Reader(['th', 'en'], gpu=True)

#### API key

In [15]:
load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
print("API Key:", api_key)

client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

API Key: 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


In [16]:
load_dotenv(override=True)
openai.api_key = os.getenv("OPENAI_API_KEY") 
assert openai.api_key, "Cannot find OPENAI_API_KEY in .env"
print("API Key:", api_key)

API Key: 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


In [17]:
POPPLER_PATH = r"C:/Users/Ned/Desktop/Poppler/poppler-24.08.0/Library/bin"

### Validate file format and size

In [18]:
# --- Upload and Validate ---
def Upload_And_Validate_File(filename: str, user_id: str, email: str) -> dict:
    allowed_extensions = {'.pdf', '.jpg', '.png', '.zip'}
    max_size_bytes = 25 * 1024 * 1024  # 25 MB

    if not os.path.exists(filename):
        return {"status": "error", "message": "File does not exist."}

    ext = os.path.splitext(filename)[1].lower()
    if ext not in allowed_extensions:
        return {
            "status": "error",
            "message": "Unsupported file format. Accepted formats are .pdf, .jpg, .png, or .zip."
        }

    file_size = os.path.getsize(filename)
    if file_size > max_size_bytes:
        return {
            "status": "error",
            "message": "File is too large. File size should not exceed 25MB."
        }

    return {"status": "success", "filename": filename}

### Import file to use EasyOCR
- Change file -> img
- Correction text by AI

In [19]:
def Read_Text_From_File(file_path: str) -> str | dict:
    logging.info(f"📥 Reading file: {file_path}")
    try:
        images = convert_from_path(file_path, dpi=200,poppler_path=POPPLER_PATH)
    except Exception as e:
        logging.error(f"❌ Failed to convert PDF: {e}")
        return {"Status": "error", "Message": f"Failed to convert PDF: {e}"}

    if not images:
        return {"Status": "error", "Message": "No extractable financial information found in the document."}

    start_time_all = time.time()

    def correct_text_with_ai(text: str) -> str:
        prompt = (
            "คุณเป็นผู้ช่วยที่ช่วยแก้ไขข้อความ OCR ภาษาไทยให้ถูกต้อง "
            "(รวมถึงคำผิดจาก OCR และเว้นวรรคผิด) "
            "โดยไม่เปลี่ยนแปลงความหมายของข้อความ ตอบกลับเฉพาะข้อความที่แก้ไขแล้วเท่านั้น:\n\n"
            f"{text}"
        )
        start_time = time.time()
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        elapsed = time.time() - start_time
        logging.info(f"⏳ GPT responded in {elapsed:.2f} seconds")
        return response.choices[0].message.content.strip()

    def batch_correct_with_ai(texts: List[str]) -> List[str]:
        def correct_chunk(chunk):
            return correct_text_with_ai("\n".join(chunk)).split("\n")

        all_fixed = []
        chunks = [texts[i:i+30] for i in range(0, len(texts), 30)]

        with ThreadPoolExecutor(max_workers=6) as executor:
            results = list(executor.map(correct_chunk, chunks))

        for corrected_lines in results:
            all_fixed.extend(corrected_lines)

        return all_fixed

    def process_page(pg_img_tuple: Tuple[int, np.ndarray]) -> str:
        pg, img = pg_img_tuple
        start_time = time.time()
        logging.info(f"🟡 Processing Page {pg}")

        results = reader.readtext(np.array(img))
        texts = [raw_text for _, raw_text, _ in results]
        logging.info(f"🔍 Found {len(texts)} text items on Page {pg}")

        if not texts:
            return f"\n📄 Page {pg}\n⚠️ No text found\n"

        fixed_texts = batch_correct_with_ai(texts)

        text_output = f"\n📄 Page {pg}\n"
        for raw_text, fixed_text in zip(texts, fixed_texts):
            text_output += f"❌ {raw_text}\n"
            text_output += f"✅ {fixed_text}\n\n"

        elapsed = time.time() - start_time
        logging.info(f"✅ Finished Page {pg} in {elapsed:.2f} seconds")
        return text_output

    with ThreadPoolExecutor(max_workers=6) as executor:
        outputs = list(executor.map(process_page, enumerate(images, start=1)))

    total_text = "".join(outputs).strip()
    elapsed_total = time.time() - start_time_all

    if not re.search(r"✅ .+", total_text):
        return {"Status": "error", "Message": "No extractable financial information found in the document."}

    logging.info("✅ All done. Output saved to memory (not file)")
    logging.info(f"⏱️ Total processing time: {elapsed_total:.2f} seconds")

    return total_text

### Extract key field by use AI

In [20]:
def Interpret_Financial_Fields(ocr_text: str) -> list | dict:
    system_prompt = (
        "You are an AI assistant that extracts key information from Thai government "
        "financial documents using OCR text. Please extract the following fields and "
        "return only in JSON format:\n\n"
        "- bill_number: เลขที่ใบขอซื้อ เช่น 10778\n"
        "- bill_type: ประเภทเอกสาร เช่น รายงานขออนุมัติจัดจ้าง\n"
        "- supplier_name: หน่วยงานหรือชื่อผู้ขาย หรือ ชื่อซัพพลายเออร์\n"
        "- amount: ยอดรวมสุทธิที่อยู่ใกล้คำว่า 'รวมทั้งสิ้น', 'ยอดรวม', 'รวมจำนวนเงิน', 'รวม'\n"
        "- payment_date: วันที่ใด ๆ ในเอกสาร\n"
        "- signature: ชื่อจริงนามสกุลที่อยู่ใกล้คำว่า 'อนุมัติ' หรือ 'ผู้เบิก'\n\n"
        "If any field is not found, use null. Respond in JSON format only without any explanation."
    )

    def extract_fields(text: str) -> dict:
        user_prompt = f"OCR Text in Thai:\n{text}\n\nPlease return the result in JSON only."
        try:
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0
            )
        except Exception as e:
            print(f"❌ AI service error: {e}")
            return {"status": "error", "message": "AI service unavailable."}

        content = response.choices[0].message.content.strip()
        print(f"📥 AI response:\n{content}\n")

        try:
            data = json.loads(content)
            if not any(data.values()):
                return {"status": "error", "message": "No extractable financial information found in the document."}
            return data
        except json.JSONDecodeError as e:
            print(f"❌ JSON parsing failed: {e}")
            return {"status": "error", "message": "AI service unavailable."}

    def extract_fields_page(args):
        """Wrapper รับ tuple (page_index, page_text) แล้วคืน dict พร้อมใส่ key 'page'"""
        idx, page_text = args
        print(f"🔍 Extracting from Page {idx}...")
        result = extract_fields(page_text)
        if not isinstance(result, dict) or result.get("status") == "error":
            print(f"⚠️ Page {idx} extraction error: {result.get('message', 'unknown')}")
            return None
        result["page"] = idx
        return result

    # แยกข้อความเป็นแต่ละหน้า
    pages = re.split(r"\n📄 Page \d+\n", ocr_text)
    pages = [p.strip() for p in pages if p.strip()]

    if not pages:
        error_msg = {"status": "error", "message": "No pages detected in OCR text."}
        print(f"⚠️ {error_msg['message']}")
        return error_msg

    # เตรียมงานขนาน
    tasks = list(enumerate(pages, start=1))
    all_results = []

    # รัน extract ข้ามหน้าแบบขนาน
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(extract_fields_page, t): t for t in tasks}
        for fut in as_completed(futures):
            res = fut.result()
            if res:
                all_results.append(res)

    # เรียงผลตามเลขหน้า
    all_results.sort(key=lambda x: x["page"])

    if not all_results:
        error_msg = {"status": "error", "message": "No extractable financial information found in the document."}
        print(f"⚠️ {error_msg['message']}")
        return error_msg

    # แสดงผลลัพธ์ทั้งหมด
    print("\n📑 Extracted All Financial Data:")
    for receipt in all_results:
        print(json.dumps(receipt, ensure_ascii=False, indent=2))

    return all_results

In [21]:
# --- Main workflow ---
if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)

    folder_path = "./"
    pdf_pattern = re.compile(r".+\.pdf$", re.IGNORECASE)

    pdf_files = [f for f in os.listdir(folder_path) if pdf_pattern.match(f)]

    logging.info("Start processing")
    if len(pdf_files) == 0:
        logging.error("No PDF files found")
    else:
        pdf_file = pdf_files[0]  
        logging.info(f"Using file: {pdf_file}")

        # 1. Upload and validate
        result = Upload_And_Validate_File(pdf_file, user_id="user123", email="user@example.com")
        if result["status"] != "success":
            logging.error(f"Error: {result['message']}")
        else:
            logging.info(f"File validated successfully: {result['filename']}")

            # 2. OCR and text correction
            ocr_text = Read_Text_From_File(result["filename"])
            if isinstance(ocr_text, dict) and ocr_text.get("Status") == "error":
                logging.error(f"Error: {ocr_text['Message']}")
            else:
                logging.info("OCR and correction done. Starting financial data extraction...")

                # 3. Extract financial fields
                extracted_data = Interpret_Financial_Fields(ocr_text)
                logging.info(f"Extracted Financial Data:\n{json.dumps(extracted_data, ensure_ascii=False, indent=2)}")

INFO:root:Start processing
INFO:root:Using file: bill.pdf
INFO:root:File validated successfully: bill.pdf
INFO:root:📥 Reading file: bill.pdf
INFO:root:🟡 Processing Page 1
INFO:root:🟡 Processing Page 2
INFO:root:🟡 Processing Page 3
INFO:root:🔍 Found 103 text items on Page 2
INFO:root:🔍 Found 89 text items on Page 1
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:🔍 Found 79 text items on Page 3
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "H

🔍 Extracting from Page 1...
🔍 Extracting from Page 2...
🔍 Extracting from Page 3...


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


📥 AI response:
{
    "bill_number": "10778",
    "bill_type": "รายงานขอความเห็นชอบและขออนุมัติจัดซื้อ/จัดจ้าง",
    "supplier_name": "มหาวิทยาลัยเชียงใหม",
    "amount": null,
    "payment_date": "31 มกราคม 2568",
    "signature": "(นางสาวจงลักษณ์ สมราง)"
}

📥 AI response:
{
    "bill_number": "10778",
    "bill_type": "รายงานขอความเห็นชอบและขออนุมัติจัดซื้อ/จัดจ้าง",
    "supplier_name": "แวนชาส ฺทราเวล โดยนางสาวอรทัย ไชยวงศ์",
    "amount": "2,200.00",
    "payment_date": "31 มกราคม 2568",
    "signature": "นายคณากร ขยัน"
}



INFO:root:Extracted Financial Data:
[
  {
    "bill_number": "168001084",
    "bill_type": "ใบสำคัญการตั้งหนี้",
    "supplier_name": "แวนชาส ฺทราเวล โดย นางสาวอรทัย",
    "amount": "2,200.00",
    "payment_date": "มี.ค. 2568",
    "signature": "วรวิชญฺจันทรฉาย",
    "page": 1
  },
  {
    "bill_number": "10778",
    "bill_type": "รายงานขอความเห็นชอบและขออนุมัติจัดซื้อ/จัดจ้าง",
    "supplier_name": "แวนชาส ฺทราเวล โดยนางสาวอรทัย ไชยวงศ์",
    "amount": "2,200.00",
    "payment_date": "31 มกราคม 2568",
    "signature": "นายคณากร ขยัน",
    "page": 2
  },
  {
    "bill_number": "10778",
    "bill_type": "รายงานขอความเห็นชอบและขออนุมัติจัดซื้อ/จัดจ้าง",
    "supplier_name": "มหาวิทยาลัยเชียงใหม",
    "amount": null,
    "payment_date": "31 มกราคม 2568",
    "signature": "(นางสาวจงลักษณ์ สมราง)",
    "page": 3
  }
]


📥 AI response:
{
    "bill_number": "168001084",
    "bill_type": "ใบสำคัญการตั้งหนี้",
    "supplier_name": "แวนชาส ฺทราเวล โดย นางสาวอรทัย",
    "amount": "2,200.00",
    "payment_date": "มี.ค. 2568",
    "signature": "วรวิชญฺจันทรฉาย"
}


📑 Extracted All Financial Data:
{
  "bill_number": "168001084",
  "bill_type": "ใบสำคัญการตั้งหนี้",
  "supplier_name": "แวนชาส ฺทราเวล โดย นางสาวอรทัย",
  "amount": "2,200.00",
  "payment_date": "มี.ค. 2568",
  "signature": "วรวิชญฺจันทรฉาย",
  "page": 1
}
{
  "bill_number": "10778",
  "bill_type": "รายงานขอความเห็นชอบและขออนุมัติจัดซื้อ/จัดจ้าง",
  "supplier_name": "แวนชาส ฺทราเวล โดยนางสาวอรทัย ไชยวงศ์",
  "amount": "2,200.00",
  "payment_date": "31 มกราคม 2568",
  "signature": "นายคณากร ขยัน",
  "page": 2
}
{
  "bill_number": "10778",
  "bill_type": "รายงานขอความเห็นชอบและขออนุมัติจัดซื้อ/จัดจ้าง",
  "supplier_name": "มหาวิทยาลัยเชียงใหม",
  "amount": null,
  "payment_date": "31 มกราคม 2568",
  "signature": "(นางสาวจงลักษณ์ สมราง)",
  "page": 3